This notebook is used to document all operations required to perform a full cycle of the CPD injection–recovery experiment for a given gap in a protoplanetary disk. Our goal is to assess the detectability of circumplanetary disks within these gaps. The content compiled here will serve as the basis for the input .py files executed on the cluster. Ultimately, the aim is to obtain the recovery–fraction curve for each gap and planet-kink location, providing further constraints for the forward-modelling predictions of CPD properties.

# Yesterday's Problem

Problem 1 to be addressed: 

1. The SPW shape of the AA_Tau original measurement set is shown as the following plot.

```python
from casatools import table
import numpy as np

out_ms = r"d/mnt/exoALMA_disk_data/data/AA_Tau_time_ave_continuum.ms"

spw_tb = table()
spw_tb.open(out_ms + "/SPECTRAL_WINDOW")

# Number of channels for each SPW
nchan = spw_tb.getcol("NUM_CHAN")

# REF_FREQUENCY column = central / reference frequency of SPW (Hz)
ref_freq = spw_tb.getcol("REF_FREQUENCY")

spw_tb.close()

# Print summary
print("SPW | nchan | ref_freq (GHz)")
for spw in range(len(nchan)):
    print(f"{spw:3d} | {nchan[spw]:5d} | {ref_freq[spw]/1e9:10.6f}")
```

```python
#-----------InjectLoop--------------#
# load the visibility data
dat = np.load('data/'+target+'_data.vis.npz')
u, v, vis, wgt = dat['u'], dat['v'], dat['Vis'], dat['Wgt']
# vis.shape = (Nrows,)


#-------------IMPORTMS----------------#
# load the model visibilities
mdl = (np.load(modelfile+'.npz'))['V']
# replace with the model visibilities (equal in both polarizations)
data[:, :, unflagged] = mdl
# data.shape = (Nrows, Npol, Nchan)
```


In my continuum MS, the SPW table lists multiple SPWs with various channel counts, but the DATA column in the MAIN table has shape (2, 1, 2,456,010). Does this mean that only the SPWs with a single averaged channel are actually used to construct this MS, and the others exist only as unused metadata? In other words, is the MS effectively single-channel because only those one-channel SPWs appear in the MAIN table? And if so, does that imply that during the import-MS step I can only inject the model visibilities into rows whose shapes match this (2, 1, Nrows) structure?

Also, looking into the way that frank recommends using uvplot in CASA to export an MS into uv tables, the function can only handle MS tables where all SPWs have the same number of channels. From exoALMA IV, they also mention in Section 2 that the visibilities in the continuum spectral windows were spectrally averaged down to one channel for the continuum analysis.

So I think it’s okay if I don’t work on the original MS but instead use a split version with spectral averaging, since I’m not sure whether the multi-channel MS can even be converted into the .npz format (in my current workflow, vis becomes a 1D array after loading with NumPy). CASA also throws an error when trying to extract the DATA column from an MS with mixed channel counts: "RuntimeError: ArrayColumn::getColumn cannot be done for column DATA; the array shapes"


In [8]:
import numpy as np

# Specific to CPD injection pipeline
target = 'AA_Tau'
# Use raw string to avoid escape sequence issues
dat = np.load(rf'D:\exoALMA_disk_data\data\{target}_time_ave_continuum.vis.npz')

print("Arrays in NPZ file:", dat.files)
print("\nArray shapes:")
for array_name in dat.files:
    print(f"  {array_name}: {dat[array_name].shape}")

print(f"\nVisibility data for {target}:")
print(f"Total visibility points: {len(dat['Vis'])}")

vis = dat['Vis']
print("vis dtype:", vis.dtype)

Arrays in NPZ file: ['u', 'v', 'Vis', 'Wgt']

Array shapes:
  u: (2456010,)
  v: (2456010,)
  Vis: (2456010,)
  Wgt: (2456010,)

Visibility data for AA_Tau:
Total visibility points: 2456010
vis dtype: complex128


Another thing I will try to do, is to do the altered export_ms in one ms file and inspect the npz file .

```python
cd /mnt/d/CPD_MPIA_Injection_Recovery_trial_fBf/DSHARP_source_code
msfile = "d/mnt/exoALMA_disk_data/data/AA_Tau_time_ave_continuum.ms"
outfile = "AA_Tau_spec_avg_data.vis.npz"

exec(open("export_ms_NONspectralavg.py").read())

```

#

# Today's Work 

1. Duplicate `/data` folder → `D:\exoALMA_disk_data\measurement_set`   
2. Split measurement set → spectrally-averaged measurement set
3. Create UV table from measurement set → `.npz` format, using uvplot inside CASA
```python
# CASA split function for all disks
cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/AA_Tau_robust2_0_gap0
/usr/local/bin/CASA/casa-6.6.1-17-pipeline-2024.1.0.8/bin/casa
execfile('spavg_ms.py', globals()) # to run spectral averaging

execfile('check_flags.py', globals())  # Check flagged data fraction
execfile('check_spw_channels.py', globals())  # final check for SPW

execfile('ms_to_npz.py', globals()) # convert the spavg ms to npz for fra

execfile('AA_Tau_robust2_0_gap0_injectloop.py', globals()) 



```python

import subprocess
import sys

# Install uvplot using CASA's pip
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'uvplot'])

# This splits the file into separate arrays immediately
u, v, vis, wgt = np.loadtxt('visibility_data.txt', unpack=True)
# u.shape = (1000,), v.shape = (1000,), etc. - separate 1D arrays

# Saves as FOUR separate arrays with names
np.savez('output.npz', u=u, v=v, Vis=vis, Wgt=wgt)

```

5. run an injection loop for one disk , using 3 flux bin
6. run the custom masking scripts